In [22]:
import os
import subprocess
import re
import pandas as pd
from IPython.display import Markdown, display

In [23]:
def run_bash_cmd(cmd, cwd):
    """Executes a bash command in the specified directory and returns the string output"""
    try:
        return subprocess.check_output(cmd, shell=True, cwd=cwd, text=True).strip()
    except subprocess.CalledProcessError:
        return "0" if "wc -l" in cmd else ""

def get_runtime_stats(cwd, filename):
    """Executes the runtime grep command and parses the hardware metrics"""
    cmd = f'grep -E "Elapsed|Maximum resident|Percent of CPU|User time|System time" {filename}'
    output = run_bash_cmd(cmd, cwd)
    
    stats = {
        "User Time (s)": 0.0,
        "System Time (s)": 0.0,
        "CPU Usage (%)": 0,
        "Elapsed Time (mm:ss)": "0:00",
        "Elapsed Time (Seconds)": 0.0,
        "Max RAM Used (KB)": 0
    }
    
    # Extract numbers from the GNU time output format
    user_time = re.search(r"User time \(seconds\):\s+([\d\.]+)", output)
    sys_time = re.search(r"System time \(seconds\):\s+([\d\.]+)", output)
    cpu_usage = re.search(r"Percent of CPU this job got:\s+(\d+)%", output)
    elapsed = re.search(r"Elapsed \(wall clock\) time \(h:mm:ss or m:ss\):\s+([\d:\.]+)", output)
    memory = re.search(r"Maximum resident set size \(kbytes\):\s+(\d+)", output)
    
    if user_time: stats["User Time (s)"] = float(user_time.group(1))
    if sys_time: stats["System Time (s)"] = float(sys_time.group(1))
    if cpu_usage: stats["CPU Usage (%)"] = int(cpu_usage.group(1))
    if memory: stats["Max RAM Used (KB)"] = int(memory.group(1))
    
    if elapsed: 
        stats["Elapsed Time (mm:ss)"] = elapsed.group(1)
        parts = stats["Elapsed Time (mm:ss)"].split(':')
        # Convert the formatted time string into raw seconds for mathematical comparison
        if len(parts) == 2:
            stats["Elapsed Time (Seconds)"] = int(parts[0]) * 60 + float(parts[1])
        elif len(parts) == 3:
            stats["Elapsed Time (Seconds)"] = int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
            
    return stats

In [24]:
standard_dir = os.path.expanduser("shadow/examples/docs/benchmarks_zkp_standard")
zkp_dir = os.path.expanduser("shadow/examples/docs/benchmarks_zkp_tor")

standard_success_cmd = 'grep -Ri "stream-success\|stream complete\|download complete\|success" shadow.data | wc -l'
standard_error_cmd = 'grep -Ri "stream-error\|error\|failed\|timeout" shadow.data | wc -l'

standard_sys = get_runtime_stats(standard_dir, "standard_run_time.txt")

zkp_success_cmd = 'grep -R "FFS-ZKP handshake successful" shadow.data | wc -l'
zkp_error_cmd = 'grep -R "FAILED\|Cheating detected\|ZKP failed\|handshake failed" shadow.data | wc -l'

zkp_success = int(run_bash_cmd(zkp_success_cmd, zkp_dir))
zkp_errors = int(run_bash_cmd(zkp_error_cmd, zkp_dir))
zkp_sys = get_runtime_stats(zkp_dir, "zkp_run_time.txt")

operations = zkp_success + zkp_errors

data = {
    "Metric": [
        "Operations",
        "User Time (s)",
        "System Time (s)",
        "CPU Usage (%)",
        "Elapsed Time (mm:ss)",
        "Elapsed Time (Seconds)",
        "Max RAM Used (KB)"
    ],
    "Standard Tor": [
        operations,
        standard_sys["User Time (s)"],
        standard_sys["System Time (s)"],
        standard_sys["CPU Usage (%)"],
        standard_sys["Elapsed Time (mm:ss)"],
        standard_sys["Elapsed Time (Seconds)"],
        standard_sys["Max RAM Used (KB)"]
    ],
    "ZKP-Tor": [
        operations,
        zkp_sys["User Time (s)"],
        zkp_sys["System Time (s)"],
        zkp_sys["CPU Usage (%)"],
        zkp_sys["Elapsed Time (mm:ss)"],
        zkp_sys["Elapsed Time (Seconds)"],
        zkp_sys["Max RAM Used (KB)"]
    ]
}
summary_df = pd.DataFrame(data)

# Calculate the performance factor
summary_df["Factor (ZKP / Tor)"] = summary_df.apply(
    lambda row: f"{(row['ZKP-Tor'] / row['Standard Tor']):.2f}x" 
    if isinstance(row['Standard Tor'], (int, float)) and row['Standard Tor'] > 0 else "N/A", 
    axis=1
)
display(Markdown(summary_df.to_markdown(index=False)))

| Metric                 | Standard Tor   | ZKP-Tor   | Factor (ZKP / Tor)   |
|:-----------------------|:---------------|:----------|:---------------------|
| Operations             | 5000           | 5000      | 1.00x                |
| User Time (s)          | 2119.76        | 2089.08   | 0.99x                |
| System Time (s)        | 3313.37        | 3197.63   | 0.97x                |
| CPU Usage (%)          | 768            | 760       | 0.99x                |
| Elapsed Time (mm:ss)   | 11:46.71       | 11:35.20  | N/A                  |
| Elapsed Time (Seconds) | 706.71         | 695.2     | 0.98x                |
| Max RAM Used (KB)      | 179612         | 177504    | 0.99x                |